![interpreto_banner](../assets/img/interpreto_banner.png)

# Classification Concept-based Explanation Tutorial

Welcome to this tutorial, our will be to obtain concept-based explanations starting from the beginning.

There are five key steps for concepts based explanations:

1. [**Split** your model in two parts](#split)
2. [Compute a dataset of **activations**](#activations)
3. [**Fit** a concept model on activations](#fit)
4. [Find the globally **important** concepts](#important)
5. [**Interpret** the concept dimensions](#interpret)

On which we add two bonus steps:

6. [**Locally** important concepts](#locally)
7. [**Evaluate** concept-based explanations](#evaluate)

*Author: Antonin Poché*

In [1]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. **Split** your model in two parts <a class="anchor" id="split"></a>

In [2]:
from transformers import AutoModelForSequenceClassification

from interpreto import ModelWithSplitPoints

splitted_model = ModelWithSplitPoints(
    model_or_repo_id="textattack/distilbert-base-uncased-ag-news",
    automodel=AutoModelForSequenceClassification,
    split_points=5,  # split at the fifth layer
    device_map="cuda",
    batch_size=1024,
)

## 2. Compute a datasets of **activations** <a class="anchor" id="activations"></a>

In [3]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")
inputs = dataset["train"]["text"][:10000]
classes_names = dataset["train"].features["label"].names

granularity = ModelWithSplitPoints.activation_granularities.CLS_TOKEN

activations = splitted_model.get_activations(
    inputs=inputs,
    activation_granularity=granularity,
    tqdm_bar=True,
    include_predicted_classes=True,
)

Computing activations: 100%|██████████| 10/10 [00:14<00:00,  1.43s/batch]


## 3. **Fit** a concept model on activations <a class="anchor" id="fit"></a>

In [4]:
from interpreto.concepts import ICAConcepts

concept_explainer = ICAConcepts(splitted_model, nb_concepts=50, device="cuda")

concept_explainer.fit(activations)

## 4. Find the globally **important** concepts <a class="anchor" id="important"></a>

In [5]:
gradients = concept_explainer.concept_output_gradient(
    inputs=inputs,
    targets=None,  # None means all classes
    activation_granularity=granularity,
    concepts_x_gradients=True,
    batch_size=64,
)

print(f"{len(gradients)=}")
print(f"{gradients[0].shape=}")

gradients = torch.stack(gradients, axis=1).squeeze()  # (num_classes, num_samples, num_concepts)
print(f"{gradients.shape=}")

Computing gradients: 100%|██████████| 157/157 [01:09<00:00,  2.26batches/s]

len(gradients)=10000
gradients[0].shape=torch.Size([4, 1, 200])
gradients.shape=torch.Size([4, 10000, 200])


In [6]:
import torch

mean_gradients = gradients.abs().mean(dim=1)
normalized_mean_gradients = mean_gradients / mean_gradients.sum(dim=1, keepdim=True)
order = torch.argsort(normalized_mean_gradients, descending=True)

for target in range(order.shape[0]):
    print(f"\nMost important concepts for target {target}:")
    for i in range(10):
        print(f"\t{order[target, i]}: {round(normalized_mean_gradients[target][order[target, i]].item(), 3)}")


Most important concepts for target 0:
	19: 0.03
	137: 0.021
	183: 0.02
	62: 0.019
	118: 0.017
	55: 0.017
	3: 0.017
	160: 0.015
	145: 0.014
	49: 0.014

Most important concepts for target 1:
	39: 0.026
	5: 0.023
	180: 0.023
	159: 0.022
	158: 0.019
	104: 0.019
	170: 0.017
	173: 0.016
	80: 0.016
	147: 0.016

Most important concepts for target 2:
	38: 0.023
	13: 0.022
	4: 0.018
	157: 0.017
	6: 0.016
	149: 0.016
	21: 0.015
	172: 0.014
	35: 0.014
	96: 0.014

Most important concepts for target 3:
	68: 0.023
	10: 0.019
	31: 0.017
	1: 0.017
	163: 0.016
	27: 0.016
	109: 0.015
	67: 0.015
	97: 0.015
	39: 0.014


In [7]:
important_concept_indices = order[:, :10].flatten().unique().tolist()
print(f"\n{important_concept_indices=}")


important_concept_indices=[1, 3, 4, 5, 6, 10, 13, 19, 21, 27, 31, 35, 38, 39, 49, 55, 62, 67, 68, 80, 96, 97, 104, 109, 118, 137, 145, 147, 149, 157, 158, 159, 160, 163, 170, 172, 173, 180, 183]


## 5. **Interpret** the concept dimensions <a class="anchor" id="interpret"></a>

In [8]:
from interpreto.concepts.interpretations import TopKInputs

topk_inputs_method = TopKInputs(
    concept_explainer=concept_explainer,
    k=10,
    activation_granularity=granularity,
    concept_encoding_batch_size=2**12,  # 4096
    use_unique_words=True,
    unique_words_kwargs={
        "count_min_threshold": round(len(inputs) * 0.001),  # appear in at least 0.1% of the samples
        "lemmatize": True,
        "words_to_ignore": [],
    },
)

In [9]:
topk_words = topk_inputs_method.interpret(
    inputs=inputs,
    concepts_indices=important_concept_indices,
)

In [10]:
for concept_id, words_importance in topk_words.items():
    print(f"Concept {concept_id}: {list(words_importance.keys()) if words_importance is not None else 'None'}")

Concept 1: ['apple', 'ipod', '64-bit', 'gb', 'pentium', 'amd', 'processor', 'motorola', 'battery', 'dell']
Concept 3: ['cabinet', 'by-election', 'majority', 'sen.', 'parliament', 'governor', 'reign', 'senate', 'ministry', 'electoral']
Concept 4: ['citigroup', 'asset', 'kofi', 'liable', '//www.investor.reuters.com/fullquote.aspx', 'fuel', 'waste', 'expense', 'invest', 'portfolio']
Concept 5: ['oct.', 'madrid', 'sept.', 'aug.', 'td', 'decathlon', 'co.', '1.7', 'indy', 'war-torn']
Concept 6: ['expo', 'corp.', 'bred', 'hd.n', 'kodak', 'aug.', '1980s', '4x100', 'jakarta', 'td']
Concept 10: ['vault', 'wis.', 'procurement', 'washingtonpost.com', 'copyright', 'amazon.com', 'appliance', 'durable', 'brain', 'siliconvalley.com']
Concept 13: ['//www.investor.reuters.com/fullquote.aspx', 'target=/stocks/quickinfo/fullquote', 'prime', 'shoot', 'rout', 'margin', 'lift', 'save', 'climb', 'defend']
Concept 19: ['...', 'us-led', 'n.', '.', '7.2', 'kenya', '1.8', 'belarus', 'delegation', ';']
Concept 21:

In [11]:
for target in range(order.shape[0]):
    print(f"\nMost important concepts for target {classes_names[target]}:")
    for i in range(10):
        words_importance = topk_words.get(order[target, i].item(), None)
        if words_importance is not None:
            print(f"\t{order[target, i]}: {list(words_importance.keys())}")
        else:
            print(f"\t{order[target, i]}: None")


Most important concepts for target World:
	19: ['...', 'us-led', 'n.', '.', '7.2', 'kenya', '1.8', 'belarus', 'delegation', ';']
	137: ['darfur', 'rwanda', 'rwandan', 'nigerian', 'somalia', 'sudanese', 'tutsi', 'chad', 'sudan', 'burundi']
	183: ['supermarket', 'w.', 'rally', 'oval', 'automaker', 'maoist', 'cleveland', 'bhp', 'trucker', 'rover']
	62: ['oct.', 'holy', 'pound', 'inflationary', 'trucker', 'muqtada', 'wage', 'td', 'doping', 'quarterback']
	118: ['birth', 'month', '0.1', 'birthday', 'worker', 'african', 'ohio', 'clothing', 'titan', 'toronto']
	55: ['matthew', 'spa', 'semi-final', 'staged', 'overnight', 'fda', 'aaron', 'iraq', 'tibet', 'shower']
	3: ['cabinet', 'by-election', 'majority', 'sen.', 'parliament', 'governor', 'reign', 'senate', 'ministry', 'electoral']
	160: ['al-qaeda', 'qaeda', 'cia', 'intelligence', 'terrorism', 'terrorist', 'osama', 'spy', 'maoist', 'laden']
	145: ['cooperation', 'convention', 'us-led', 'un', 'peace', 'tri-nations', 'arbitration', 'conflict',

## 5. bis class-wise

In [12]:
activations.keys()

dict_keys(['distilbert.transformer.layer.5', 'predictions'])

In [13]:
concept_explainers = {}
concept_interpretations = {}
concept_importances = {}

for target, class_name in enumerate(classes_names):
    print(f"\nComputing concepts for class: {class_name}")
    indices = (activations["predictions"] == target).nonzero(as_tuple=True)[0]
    class_wise_inputs = [inputs[i] for i in indices]
    class_wise_activations = {k: v[indices] for k, v in activations.items()}

    print(f"\tThere are {len(class_wise_inputs)} samples. Training ICA...")

    concept_explainer = ICAConcepts(splitted_model, nb_concepts=20, device="cuda")
    concept_explainer.fit(class_wise_activations)
    concept_explainers[target] = concept_explainer

    print("\tComputing concepts importance...")
    gradients = concept_explainer.concept_output_gradient(
        inputs=class_wise_inputs,
        targets=[target],
        activation_granularity=granularity,
        concepts_x_gradients=True,
        batch_size=64,
    )
    gradients = torch.stack(gradients, axis=1).squeeze()  # (num_samples, num_concepts)

    mean_gradients = gradients.abs().mean(dim=0)
    normalized_mean_gradients = mean_gradients / mean_gradients.sum(dim=0, keepdim=True)
    order = torch.argsort(normalized_mean_gradients, descending=True)
    concept_importances[target] = normalized_mean_gradients

    # important_concept_indices = order[:10].flatten().unique().tolist()

    print("\tInterpreting important concepts...")
    topk_inputs_method = TopKInputs(
        concept_explainer=concept_explainer,
        k=10,
        activation_granularity=granularity,
        concept_encoding_batch_size=2**12,  # 4096
        use_unique_words=True,
        unique_words_kwargs={
            "count_min_threshold": round(len(class_wise_inputs) * 0.002),  # appear in at least 0.2% of the samples
            "lemmatize": True,
            "words_to_ignore": [],
        },
    )

    topk_words = topk_inputs_method.interpret(
        inputs=class_wise_inputs,
        concepts_indices="all",  # all concepts
    )
    concept_interpretations[target] = topk_words

    print(f"\tMost important concepts for target {class_name}:")
    for concept_id in order[:10]:
        words_importance = topk_words.get(concept_id.item(), None)
        if words_importance is not None:
            print(f"\t\t{concept_id}: {list(words_importance.keys())}")
        else:
            print(f"\t\t{concept_id}: None")


Computing concepts for class: World
	There are 2456 samples. Training ICA...
	Computing concepts importance...


Computing gradients: 100%|██████████| 39/39 [00:14<00:00,  2.64batches/s]


	Interpreting important concepts...
	Most important concepts for target World:
		7: ['ivory', 'to\\protect', 'eclipsed', '\\', 'doe', 'venus', 'greenpeace', 'wasteland', 'scala', 'spar']
		19: ['rationed', 'per-share', 'deficit', 'finance', 'economy', 'inflation', 'wage', 'cash', 'surcharge', 'circulation']
		15: ['nuclear', 'smuggling', 'erupt', 'al-qaeda-linked', 'u.s.-backed', 'influenza', 'hostage-taking', 'u.s.-led', 'abducted', 'kidnapped']
		1: ['sanofi-aventis', 'pre-poll', 'twickenham', 'first-half', 'goodale', 'oakland', 'top-seeded', 'gasoline', 'pinch-hitter', 'al-beshir']
		8: ['olympic-record', 'six-nation', 'heptathlon', 'phelps', 'top-seeded', 'first-half', 'career-high', '200-meter', 'gold-medal', 'record-low']
		13: ["shi'ite\\militiamen", 'bustos', 'fla.', 'milosevic', 'loyalist', 'ex-paramilitary', 'rebel-held', 'blockaded', 'ex-rebel', 'cromwell']
		18: ['peshawar', 'liberate', 'blitz', 'hunger-striking', 'war-ravaged', 'al-qaeda', 'qaeda-linked', 'war-torn', 'mehd

/home/antonin.poche/interpreto/.venv/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:127: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


	Computing concepts importance...


...

	Interpreting important concepts...
	Most important concepts for target Sports:
		2: ['ioc', 'expo', 'nato', 'squash', 'delegation', 'badminton', 'judo', 'archery', 'weightlifting', 'fencing']
		6: ['olympics-henin', 'olympics-federer', 'seventh-seeded', 'west-leading', 'olympic-record', 'fifth-seeded', 'league-leading', 'pac-10', 'villarreal', 'second-seed']
		10: ['mamaroneck', 'calif.', 'barclays', 'cash', 'sbc', 'dividend', 'finance', 'pound', 'meares', 'cent']
		4: ['nba', 'pac-10', 'ronaldo', '36-hole', 'nhl', 'second-half', 'all-american', 'payton', 'pacer', 'goaltender']
		8: ['800-meter', '10,000-meter', '4,000-meter', 'middle-distance', '400-meter', 'iaaf', '3000m', '800m', '10,000m', '400m']
		17: ['pound', 'anastasia', 'jinx', 'baghdad', 'snatch', 'bustos', 'sept.', 'hee-sham', 'haile', 'saulnier']
		5: ['punter', 'shot-putters', 'five-shot', '12-yard', 'infielder', 'pound', 'four-shot', '5-yard', 'two-shot', 'dressage']
		9: ['infielder', 'nba', 'stats', 'shortstop', 'pres

Computing gradients: 100%|██████████| 38/38 [00:15<00:00,  2.52batches/s]


	Interpreting important concepts...
	Most important concepts for target Business:
		15: ['second-straight', '1st-half', '2nd-half', 'first-half', 'second-half', 'first-quarter', '2nd-quarter', 'second-quarter', 'jones/ap', 'td']
		6: ['overtime', '2nd-quarter', 'first-quarter', 'fourth-quarter', 'second-quarter', 'yukos', '1st-half', 'third-quarter', '2nd-half', 'atlanta']
		8: ['deli-style', 'multibillion-dollar', 'credit-card', 'euro', 'ex-enron', 'enron', 'nyse', 'kmart', 'paycheck', 'eurozone']
		5: ['profit-taking', 'multibillion-dollar', 'pay-per-view', '85/share', 'ticker=ltd.n', 'profitability', 'ticker=fdx.n', 'cnn/money', 'buffett', 'ticker=aci.n']
		7: ['petroleum', 'refinery', 'oil', 'coal', 'uranium', 'tobacco', 'oil-producing', 'iraqi', 'iraq', 'nuclear']
		17: ['home-improvement', 'calif.', 'calif', 'renovation', 'asbestos', 'housing', 'framingham', 'landmark', 'construction', 'property']
		19: ['oct.', 'aug.', 'rep.', 'toronto-dominion', 'nov.', 'macapagal-arroyo', 'sen

Computing gradients: 100%|██████████| 44/44 [00:17<00:00,  2.54batches/s]


	Interpreting important concepts...
	Most important concepts for target Sci/Tech:
		10: ['transistor', 'pentium', '802.11a', '802.11n', 'amd', 'hp-ux', 'microprocessor', 'semiconductor', '64-bit', 'supercomputer']
		14: ['calif.', 'terengganu', 'slate-colored', 'pent-up', 'anti-bush', 'sept.', 'iraq', 'islamabad', 'republican', 'methamphetamine']
		9: ['cyber-crime', 'e-government', 'p2pnet.net', 'siliconvalley.com', 'earthlink-hosted', 'pornography', 'hd-dvd', 'nanotechnology', 'macromedia', 'cybercafe']
		4: ['802.11a', '802.11n', 'forbes.com', 'flu', 'hd-dvd', 'u.k.', 'whale', 'siliconvalley.com', 'salesforce.com', 'guantanamo']
		5: ['second-quarter', 'fourth-quarter', 'nfl', 'rbot', 'crs-1', 'csco.o', 'doping', 'ap', 'discus', '15-in']
		2: ['gasoline', 'pound', 'midmarket', 'soybean', 'forbes.com', 'economy', 'gambling', 'fla.', 'cut-rate', 'cent']
		0: ['gasoline', 'soybean', 'cbs.mw', 'midmarket', 'flu', 'sewage', 'waste', 'outlook', '//ad.doubleclick.net/ad/idg.us.ifw.general/

## 6. **Locally** important concepts <a class="anchor" id="locally"></a>

In [32]:
test_examples, labels = dataset["test"]["text"][:50], dataset["test"]["label"][:50]
for i, (example, label) in enumerate(zip(test_examples, labels, strict=False)):
    print(f"Example {i}: {classes_names[label]} - {example}\n")

Example 0: Business - Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.

Example 1: Sci/Tech - The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\privately funded suborbital space flight, has officially announced the first\launch date for its manned rocket.

Example 2: Sci/Tech - Ky. Company Wins Grant to Study Peptides (AP) AP - A company founded by a chemistry researcher at the University of Louisville won a grant to develop a method of producing better peptides, which are short chains of amino acids, the building blocks of proteins.

Example 3: Sci/Tech - Prediction Unit Helps Forecast Wildfires (AP) AP - It's barely dawn when Mike Fitzpatrick starts his shift with a blur of colorful maps, figures and endless charts,

In [36]:
business_example = test_examples[41]
business = 2  # business

business_local_importance = (
    concept_explainers[business]
    .concept_output_gradient(
        inputs=[business_example],
        activation_granularity=granularity,
        concepts_x_gradients=True,
    )[0][business]
    .squeeze()
)

normalized_importance = business_local_importance.abs() / business_local_importance.abs().sum()
ordered_indices = torch.argsort(normalized_importance, descending=True)

print(f"\nbusiness example: {business_example}")
for concept_id in ordered_indices[:5]:
    importance = normalized_importance[concept_id]
    words_importance = concept_interpretations[business][concept_id.item()]
    if words_importance is not None:
        print(f"\t{concept_id}: {round(importance.item(), 3)} - {list(words_importance.keys())}")
    else:
        print(f"\t{concept_id}: {round(importance.item(), 3)} - None")

print("\n")

sport_example = test_examples[26]
sport = 1  # sports

sport_local_importance = (
    concept_explainers[sport]
    .concept_output_gradient(
        inputs=[sport_example],
        activation_granularity=granularity,
        concepts_x_gradients=True,
    )[0][sport]
    .squeeze()
)

normalized_importance = sport_local_importance.abs() / sport_local_importance.abs().sum()
ordered_indices = torch.argsort(normalized_importance, descending=True)

print(f"\nSport example: {sport_example}")
for concept_id in ordered_indices[:5]:
    importance = normalized_importance[concept_id]
    words_importance = concept_interpretations[sport][concept_id.item()]
    if words_importance is not None:
        print(f"\t{concept_id}: {round(importance.item(), 3)} - {list(words_importance.keys())}")
    else:
        print(f"\t{concept_id}: {round(importance.item(), 3)} - None")

print("\n")
sci_tech_example = test_examples[20]
sci_tech = 3  # sci/tech

sci_tech_local_importance = (
    concept_explainers[1]
    .concept_output_gradient(
        inputs=[sci_tech_example],
        activation_granularity=granularity,
        concepts_x_gradients=True,
    )[0][sci_tech]
    .squeeze()
)

normalized_importance = sci_tech_local_importance.abs() / sci_tech_local_importance.abs().sum()
ordered_indices = torch.argsort(normalized_importance, descending=True)

print(f"Sci/Tech example: {sci_tech_example}")
for concept_id in ordered_indices[:5]:
    importance = normalized_importance[concept_id]
    words_importance = concept_interpretations[sci_tech][concept_id.item()]
    if words_importance is not None:
        print(f"\t{concept_id}: {round(importance.item(), 3)} - {list(words_importance.keys())}")
    else:
        print(f"\t{concept_id}: {round(importance.item(), 3)} - None")

Computing gradients: 100%|██████████| 1/1 [00:00<00:00,  2.44batches/s]



business example: Retailers Vie for Back-To-School Buyers (Reuters) Reuters - Apparel retailers are hoping their\back-to-school fashions will make the grade among\style-conscious teens and young adults this fall, but it could\be a tough sell, with students and parents keeping a tighter\hold on their wallets.
	7: 0.126 - ['petroleum', 'refinery', 'oil', 'coal', 'uranium', 'tobacco', 'oil-producing', 'iraqi', 'iraq', 'nuclear']
	14: 0.12 - ['deli-style', 'wal-mart', 'sears', 'grocer', 'steakhouse', 'mall', 'supermarket', 'drugstore', 'starbucks', 'fast-food']
	15: 0.105 - ['second-straight', '1st-half', '2nd-half', 'first-half', 'second-half', 'first-quarter', '2nd-quarter', 'second-quarter', 'jones/ap', 'td']
	4: 0.092 - ['gm', 'daimlerchrysler', 'oldsmobile', 'auto-parts', 'auto-body', 'non-opec', 'automaker', 'volkswagen', 'ford', 'turbotax']
	3: 0.085 - ['cash-strapped', 'economy', 'mortgage\\interest', '7.5-trillion', 'asset', 'cashless', 'dividend', 'insolvency', 'stockowner', 'su

Computing gradients: 100%|██████████| 1/1 [00:00<00:00,  3.04batches/s]



Sport example: Giddy Phelps Touches Gold for First Time Michael Phelps won the gold medal in the 400 individual medley and set a world record in a time of 4 minutes 8.26 seconds.
	17: 0.206 - ['pound', 'anastasia', 'jinx', 'baghdad', 'snatch', 'bustos', 'sept.', 'hee-sham', 'haile', 'saulnier']
	2: 0.132 - ['ioc', 'expo', 'nato', 'squash', 'delegation', 'badminton', 'judo', 'archery', 'weightlifting', 'fencing']
	10: 0.115 - ['mamaroneck', 'calif.', 'barclays', 'cash', 'sbc', 'dividend', 'finance', 'pound', 'meares', 'cent']
	4: 0.102 - ['nba', 'pac-10', 'ronaldo', '36-hole', 'nhl', 'second-half', 'all-american', 'payton', 'pacer', 'goaltender']
	8: 0.091 - ['800-meter', '10,000-meter', '4,000-meter', 'middle-distance', '400-meter', 'iaaf', '3000m', '800m', '10,000m', '400m']




Computing gradients: 100%|██████████| 1/1 [00:00<00:00,  3.24batches/s]

Sci/Tech example: IBM to hire even more new workers By the end of the year, the computing giant plans to have its biggest headcount since 1991.
	6: 0.577 - ['inc.\\', '\\\\', '\\', '\\\\i', '\\the', '\\\\the', '/font', 'cent', 'font', 'thursday\\delivered']
	2: 0.158 - ['gasoline', 'pound', 'midmarket', 'soybean', 'forbes.com', 'economy', 'gambling', 'fla.', 'cut-rate', 'cent']
	17: 0.055 - ['r.i.', 'expo', 'video-game', 'cent', 'balance', 'blaster', 'gambling', 'iraq', 'finance', 'cut-rate']
	7: 0.036 - ['comet', 'jupiter-sized', 'saturnian', 'venus', 'orbiter', 'neptune', 'lunar', 'saturn', 'martian', 'asteroid']
	4: 0.031 - ['802.11a', '802.11n', 'forbes.com', 'flu', 'hd-dvd', 'u.k.', 'whale', 'siliconvalley.com', 'salesforce.com', 'guantanamo']


## 7. **Evaluate** concept-based explanations <a class="anchor" id="evaluate"></a>

### 7.1 Evaluate the concept-space from the [third part](#fit)

### 7.2 Evaluate the concepts-interpretations from the [fifth step](#important)

### 7.3 Evaluate the whole concept-based explanations with `ConSim`

In [15]:
# Define the User-LLM (the meta-predictor and llm as a judge)
user_llm = OpenAILLM(api_key="YOUR_OPENAI_API_KEY", model="gpt4o-mini")

# Initialize the ConSim  with the model with split points and the user-llm
# Therefore, a given ConSim metric can be used on different explainers for cleaner comparison
con_sim = ConSim(model_with_split_points, user_llm, classes=classes)

# Select examples for evaluation
samples, labels, predictions = con_sim.select_examples(
    dataset["train"]["text"],
    dataset["train"]["label"],
)

# Compute a baseline and ConSim score to give sense to the explainer ConSim score
baseline = con_sim.evaluate(samples, labels, predictions, prompt_type=PromptTypes.L2_baseline_with_lp)

# Compute the ConSim score for an explainer # TODO: allow to give a list
con_sim_score = con_sim.evaluate(
    samples, labels, predictions, concept_explainer, prompt_type=PromptTypes.E3_global_and_local_concepts_with_lp
)

NameError: name 'OpenAILLM' is not defined

: 